# Defect reduction in a steel rolling line — Lean / DMAIC analysis

**Author:** Esteban López · Industrial Engineer  
**Dataset:** Steel Plates Faults — UCI Machine Learning Repository (#198), Semeion Research Center. 1,941 plates, 27 process/geometry variables, 7 defect types.  
**Source:** https://archive.ics.uci.edu/dataset/198/steel+plates+faults

## Project goal (continuous-improvement focus)
Apply the **DMAIC** cycle and **Lean / Six Sigma** tools to real steel-plate inspection data to: (1) quantify and prioritise defect modes (Pareto), (2) analyse root cause, (3) build a classifier that supports automated inspection, and (4) propose countermeasures that reduce scrap and improve first-pass yield (FPY).

> **Reproducible** notebook: it downloads the full dataset from UCI. Run it top to bottom.

## 0. Setup
Requirements: `pip install ucimlrepo pandas numpy scikit-learn matplotlib`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 60)
plt.rcParams.update({'figure.figsize': (9, 5), 'font.size': 11})
NAVY, CYAN, GOLD = '#0b1f3a', '#16b3c7', '#f2c14e'

## 1. DEFINE — Load data
We download directly from the UCI repository. If offline, the notebook falls back to an optional local CSV.

In [ ]:
FAULTS = ['Pastry','Z_Scratch','K_Scatch','Stains','Dirtiness','Bumps','Other_Faults']

def load_data():
    # 1) via official UCI package
    try:
        from ucimlrepo import fetch_ucirepo
        ds = fetch_ucirepo(id=198)
        X = ds.data.features.copy()
        y = ds.data.targets.copy()
        return pd.concat([X, y], axis=1)
    except Exception as e:
        print('ucimlrepo not available:', e)
    # 2) fallback: local CSV (same format)
    return pd.read_csv('steel_plates_raw.csv')

df = load_data()
print('Shape:', df.shape)
df.head()

In [ ]:
# Build a single defect-type column from the 7 one-hot columns
fault_cols = [c for c in df.columns if c in FAULTS]
assert len(fault_cols) == 7, f'Expected 7 defect columns, got {len(fault_cols)}: {fault_cols}'
df['fault_type'] = df[fault_cols].idxmax(axis=1)
features = [c for c in df.columns if c not in fault_cols + ['fault_type']]
print(f'{len(features)} input variables | {len(fault_cols)} defect classes')

## 2. MEASURE — Data quality and baseline

In [ ]:
print('Total missing values:', int(df.isna().sum().sum()))
print('Duplicate rows:', int(df.duplicated().sum()))
counts = df['fault_type'].value_counts()
print('\nDefect distribution:')
print(counts)
print('\nImbalance ratio (max/min): %.1f' % (counts.max()/counts.min()))

## 3. ANALYZE — Pareto chart (80/20 prioritisation)
We identify the “vital few”: the defect types that hold most of the problem.

In [ ]:
pareto = counts.sort_values(ascending=False)
cum = pareto.cumsum() / pareto.sum() * 100

fig, ax1 = plt.subplots(figsize=(9, 5.2))
colors = [CYAN if c <= 80.5 else '#9fb3c8' for c in cum]
ax1.bar(pareto.index, pareto.values, color=colors, edgecolor='white')
for i, v in enumerate(pareto.values):
    ax1.text(i, v + 8, str(v), ha='center', fontweight='bold', color=NAVY, fontsize=9)
ax1.set_ylabel('Defect frequency'); ax1.set_title('Pareto of steel-plate defects', fontweight='bold', color=NAVY)
ax2 = ax1.twinx()
ax2.plot(pareto.index, cum.values, color=GOLD, marker='o', lw=2.4)
ax2.axhline(80, color='grey', ls='--'); ax2.set_ylabel('Cumulative %'); ax2.set_ylim(0, 105)
plt.xticks(rotation=20, ha='right'); plt.tight_layout(); plt.show()

vital_few = cum[cum <= 80.5].index.tolist()
print(f'Vital few (~80%): {vital_few}  ->  {cum[vital_few[-1]]:.1f}% of the total')

**Lean finding:** 3 of 7 defect types (Other_Faults, Bumps, K_Scatch) hold ~75% of all defective plates. The largest class, *Other_Faults*, is a catch-all bucket: its size points to an opportunity to improve the **definition and traceability** of failure modes (a management finding, not only a process one).

### 3.1 Defect vs steel type and geometry

In [ ]:
# Which variables discriminate the vital defects? Standardised means per class
num = df[features].select_dtypes('number')
z = (num - num.mean()) / num.std()
z['fault_type'] = df['fault_type']
profile = z.groupby('fault_type').mean().T
top_var = profile.std(axis=1).sort_values(ascending=False).head(10).index
profile.loc[top_var].round(2)

### 3.2 Statistical process control (SPC) and capability (Cpk)
We take a continuous process variable (plate thickness) to illustrate a control chart and the capability calculation. *Specification limits are illustrative; on the shop floor they come from the product routing sheet.*

In [ ]:
var = 'Steel_Plate_Thickness' if 'Steel_Plate_Thickness' in df.columns else num.std().idxmax()
x = df[var].astype(float)
mu, sd = x.mean(), x.std()
LSL, USL = mu - 3.5*sd, mu + 3.5*sd  # illustrative spec limits
Cp = (USL - LSL) / (6*sd)
Cpk = min((USL - mu), (mu - LSL)) / (3*sd)
print(f'Variable: {var}')
print(f'Mean={mu:.1f}  Sigma={sd:.1f}  Cp={Cp:.2f}  Cpk={Cpk:.2f}')

fig, ax = plt.subplots(figsize=(9,4))
ax.plot(x.values[:200], color=CYAN, lw=0.9)
for L,c,t in [(mu,NAVY,'Mean'),(mu+3*sd,'grey','+3s'),(mu-3*sd,'grey','-3s')]:
    ax.axhline(L, color=c, ls='-' if t=='Mean' else '--', lw=1)
ax.set_title(f'Control chart — {var} (first 200 plates)', color=NAVY, fontweight='bold')
ax.set_xlabel('Plate'); ax.set_ylabel(var); plt.tight_layout(); plt.show()

## 4. IMPROVE — Inspection-support classifier
A model that predicts the defect type from vision/process variables enables **automated classification**, cutting inspection time and human variability (digital poka-yoke).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

X = df[features].select_dtypes('number')
y = df['fault_type']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1, class_weight='balanced')
rf.fit(Xtr, ytr)
pred = rf.predict(Xte)
print('Accuracy: %.3f' % accuracy_score(yte, pred))
print('Macro F1: %.3f' % f1_score(yte, pred, average='macro'))
print(classification_report(yte, pred))

In [ ]:
cm = confusion_matrix(yte, pred, labels=sorted(y.unique()))
fig, ax = plt.subplots(figsize=(7,6))
ax.imshow(cm, cmap='Blues')
labels = sorted(y.unique())
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, cm[i,j], ha='center', va='center', color='white' if cm[i,j]>cm.max()/2 else 'black', fontsize=9)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title('Confusion matrix', fontweight='bold', color=NAVY)
plt.tight_layout(); plt.show()

In [ ]:
# Feature importance: what drives quality the most?
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(12)
fig, ax = plt.subplots(figsize=(8,5))
imp[::-1].plot.barh(ax=ax, color=CYAN)
ax.set_title('Top 12 variables predicting the defect type', fontweight='bold', color=NAVY)
plt.tight_layout(); plt.show()
imp

## 5. CONTROL — Conclusions and countermeasures

1. **Prioritise the vital few:** tackle Other_Faults, Bumps and K_Scatch first (~75% of the problem). Highest return per unit of effort.
2. **Improve the defect taxonomy:** *Other_Faults* is too large → redefine categories and train inspection to shrink the catch-all and enable fine root-cause analysis.
3. **SPC on critical variables:** monitor the model's highest-importance variables with control charts; react before scrap is produced.
4. **Digital poka-yoke:** deploy the classifier as inspection support to standardise the criterion and free up inspector time.
5. **Improvement loop:** retrain the model and recompute the Pareto per shift/batch to verify the effect of countermeasures (PDCA).

*Next steps:* validate real specification limits, add cost per defect type for a €/$-weighted Pareto, and connect to a shop-floor dashboard.